In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset,DataLoader,Dataset
import torch.nn.functional as F
import numpy as np
import random
import os
import pandas as pd
from torchvision.datasets.cifar import CIFAR100
import torchvision.transforms as transforms
import torchvision
import matplotlib.pyplot as plt


import tools.utils as utils
import tools.cnn_adj_matrix as build_cnn_adj
from tools.cnn import CNN
from tools.LeNet5 import LeNet
from tools.LeNet5_small import LeNet as LeNet_custom_v2
from RicciCurvature.OllivierRicci import OllivierRicci
from tools.FC_linear import FC_Linear
from tools.small_model import FC_MD
from tools.alexnet import AlexNet_CIFAR10
from tools.vgg16 import VGG16_CIFAR10

In [ ]:
seed = 53

# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
MODEL_PATH = "CNN/models/new/"

In [ ]:
# def load_dataset_from_disk(path, batch_size=128, shuffle=True, transform=None):
#     data_path = f"{path}/data.pt"
#     images, labels = torch.load(data_path)
#     dataset = TensorDataset(images, labels)
#     return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
class TransformedTensorDataset(Dataset):
    def __init__(self, tensors, transform=None):
        self.images, self.labels = tensors
        self.transform = transform

    def __getitem__(self, index):
        img = self.images[index]
        label = self.labels[index]
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.labels)

def load_dataset_from_disk(path, batch_size=128, shuffle=True, transform=None):
    data_path = f"{path}/data.pt"
    images, labels = torch.load(data_path)
    dataset = TransformedTensorDataset((images, labels), transform=transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


In [ ]:
# Data transforms for CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    # transforms.ToTensor(),
    # transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010)),
])

In [ ]:
train_loader = load_dataset_from_disk("./data/CIFAR100_train", batch_size=256, transform=transform_train)
valid_loader = load_dataset_from_disk("./data/CIFAR100_val", batch_size=2000, shuffle=True)

In [ ]:
# Load one batch from the train_loader
# images, labels = next(iter(valid_loader))  # (B, C, H, W), (B,)

# # Make a grid of 8x4 = 32 images (or fewer depending on batch size)
# grid_img = torchvision.utils.make_grid(images[:1], nrow=1, normalize=True)

# # Convert from PyTorch tensor (C, H, W) to numpy array (H, W, C)
# np_img = grid_img.permute(1, 2, 0).cpu().numpy()

# # Plot
# plt.figure(figsize=(10, 5))
# plt.imshow(np_img)
# plt.title(f"Label {labels[:1].item()}")
# plt.axis("off")
# plt.show()

In [ ]:
# train_loader = load_dataset_from_disk("./data/MNIST_train", batch_size=128, transform=transform_train)
# valid_loader = load_dataset_from_disk("./data/MNIST_val", batch_size=2000, shuffle=False)

In [ ]:

# data_train = MNIST('./data/mnist',
#                   train=True,
#                   download=True,
#                   transform=transforms.Compose([
#                       # transforms.Resize((32, 32)),
#                       transforms.ToTensor()]))

# data_test = MNIST('./data/mnist',
#                   train=False,
#                   download=True,
#                   transform=transforms.Compose([
#                       # transforms.Resize((32, 32)),
#                       transforms.ToTensor()]))

In [ ]:
# transform_train = torchvision.transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(size=32, padding=4),
#     transforms.ToTensor(),
#     # transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
# ])

# transform_test = torchvision.transforms.Compose([
#     transforms.ToTensor(),
#     # transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
# ])

# data_train = CIFAR10('./data/cifar10', train=True, download=True, transform=transform_train)
# data_test = CIFAR10('./data/cifar10', train=False, download=True, transform=transform_test)

In [ ]:
# def get_new_data(l1):

#     # selected classes
#     train_i1 = torch.tensor([i for i, (_, label) in enumerate(data_train) if label in l1])
#     test_i1 = torch.tensor([i for i, (_, label) in enumerate(data_test) if label in l1])
    
#     train_index = torch.randperm(len(train_i1))
#     valid_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[0:5000]])
#     train_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[5000:,]])
#     test_dataset = torch.utils.data.Subset(data_test, test_i1)
    
#     train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
#     test_loader = DataLoader(test_dataset, batch_size=2000, num_workers=2)
#     valid_loader = DataLoader(valid_dataset, batch_size=2000, num_workers=2)
    
#     return train_loader, test_loader, valid_loader

In [ ]:
# # selected_classes = [0,1,2,3,4,5,6,7,8,9]
# # # train_loader, test_loader, valid_loader = get_new_data(selected_classes)

model_name = "vgg16_100_ori_relu.pth"
net_H = VGG16_CIFAR10(input_c=3, num_classes=100)
net_H.load_state_dict(torch.load(MODEL_PATH + model_name))
net_H = net_H.to(device)
loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(net_H.parameters(), lr=0.1, weight_decay=0.0005)
optimizer = torch.optim.SGD(net_H.parameters(), lr=0.001, weight_decay=0.0005, momentum=0.9)

In [ ]:
# # selected_classes = [0,1,2,3,4,5,6,7,8,9]
# # train_loader, test_loader, valid_loader = get_new_data(selected_classes)
# dims = [784, 200, 150, 10]

# net_H = FC_MD(dims, 2)
# # net_H.load_state_dict(torch.load(MODEL_PATH + "best_cifar_adv.pth"))
# net_H = net_H.to(device)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(net_H.parameters(), lr=2e-3, weight_decay=0.0001)


In [ ]:
def standard_PGD(model, images, labels, eps=11/255, alpha=2/255, iters=40):
    images = images.to(device)
    labels = labels.to(device)
    loss = nn.CrossEntropyLoss()
        
    ori_images = images.data
        
    for i in range(iters) :    
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels).to(device)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()
            
    return images

In [ ]:
def test_adversarial(net, loader, eps=.1, alpha=.1, iters=100):
    # prepare model for testing (only important for dropout, batch norm, etc.)
    net.eval()
    
    correct = 0

    for data, target in loader:

        data = standard_PGD(net, data, target, eps=eps, alpha=alpha, iters=iters)
        data, target = data.to(device), target.to(device)

        output = net(data)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum()
    
    print('Test set: Avg. Accuracy: {}/{} ({:.2f}%)'.format(
        correct, len(loader.dataset),
        (100. * correct / len(loader.dataset))))
    
    return correct / len(loader.dataset)


def train_adversarial(net, net_attack, loader, optimizer, epoch, eps=.1, alpha=.1, iters=100):
    # prepare model for training (only important for dropout, batch norm, etc.)
    net.train()

    total_loss = 0
    correct = 0
    
    for batch_idx, (data, target) in enumerate(loader):
        #print(data.size())

        data = standard_PGD(net, data, target, eps=eps, alpha=alpha, iters=iters).to(device)
        target = target.to(device)
        optimizer.zero_grad()
        
        output = net(data)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum()
        
        loss = loss_fn(output, target)
        total_loss += loss

        # compute gradients and make updates
        loss.backward()
        optimizer.step()
        
    print('Adversary training set: Avg. Accuracy: {}/{} ({:.2f}%)'.format(
    correct, len(loader.dataset),
    (100. * correct / len(loader.dataset))))

    return total_loss/len(loader), correct / len(loader.dataset)


In [ ]:
# test_acc = test_adversarial(net_H, test_loader, eps=0.20, alpha=2/255, iters=40)

In [ ]:
def test(net_H, loader):
    # prepare model for testing (only important for dropout, batch norm, etc.)
    net_H.eval()

    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)

            output = net_H(data)

            pred = output.data.max(1, keepdim=True)[1]
            correct += (pred.eq(target.data.view_as(pred)).sum().item())

            total = total + 1

    return 100.0 * correct / len(loader.dataset)

In [ ]:
def train(net_H, loader):
    net_H.train()

    correct = 0
    total_loss = 0

    for batch_idx, (data, target) in enumerate(loader):
        # print(batch_idx)
        data, target = data.to(device), target.to(device)

        # clear up gradients for backprop
        optimizer.zero_grad()
        output = net_H(data)
        
        loss = loss_fn(output, target)
        total_loss += loss

        # compute gradients and make updates
        loss.backward()
        optimizer.step()

        pred = output.data.max(1, keepdim=True)[1]
        correct += (pred.eq(target.data.view_as(pred)).sum().item())

    return total_loss/len(loader)
    

In [ ]:
eps = [2/255]

best_acc = 0.

for ep in eps:
    for e in range(200):
        print(f"Epoch {e}: ")
        loss = train(net_H, train_loader)
        val_acc = test(net_H, valid_loader)
        # loss, acc = train_adversarial(net_H, net_H, train_loader, optimizer, e, eps=ep, alpha=2/255, iters=20)
        # val_acc = test_adversarial(net_H, valid_loader, eps=ep, alpha=2/255, iters=20)
        # acc = test(valid_loader)
        
        # print(f'for {e}: the clean acc = {acc}')
        
        # if (e % 10 == 0):
        #     torch.save(net_H.state_dict(), MODEL_PATH + str(e) + "_lenet.pth")
        #     print(f'For epoch {e}: the training loss is {loss} and the accuracy on validation set is {val_acc}.')
        #     print()

        if (val_acc > best_acc):
            best_acc = val_acc
            print(f'New best model saved! For epoch {e}: the training loss is {loss} and the accuracy on validation set is {val_acc}.')
            print()
            torch.save(net_H.state_dict(), MODEL_PATH + "vgg16_100_ori_relu1.pth")

